# Notebook 03 — Tipo de escola × desempenho
**Eixo de análise:** Gustavo Silveira  
**Pergunta:** Como o tipo de escola (pública vs privada) influencia o desempenho no ENEM?

Análise da variável **TP_ESCOLA** em relação às notas, com recortes por:
- Estado (RS vs SP)
- Ano (2019 vs 2023, pré e pós-pandemia)
- Área de conhecimento

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')


df_2019 = pd.read_parquet('../../dados-processados/enem2019_RS_SP.parquet')
df_2023 = pd.read_parquet('../../dados-processados/enem2023_RS_SP.parquet')
df = pd.concat([df_2019, df_2023], ignore_index=True)


presentes = (df['TP_PRESENCA_CN']==1) & (df['TP_PRESENCA_CH']==1) & \
            (df['TP_PRESENCA_LC']==1) & (df['TP_PRESENCA_MT']==1)
df = df[presentes].copy()


areas = ['NU_NOTA_CN','NU_NOTA_CH','NU_NOTA_LC','NU_NOTA_MT','NU_NOTA_REDACAO']
df['NOTA_MEDIA'] = df[areas].mean(axis=1)

print(f"Base de análise: {len(df):,} participantes presentes nos 2 dias")

## Sobre a variável TP_ESCOLA

| Código | Significado |
|---|---|
| 1 | Não respondeu |
| 2 | Pública |
| 3 | Privada |

**Atenção:** muitos participantes (que não estavam mais no ensino médio na época da prova, treineiros, etc.) marcam "Não respondeu". Para análise comparativa pública × privada, vamos filtrar apenas códigos 2 e 3.

In [ ]:

print("Distribuição total (todos os participantes presentes):")
print(df['TP_ESCOLA'].value_counts(dropna=False))

print("\nProporção:")
print((df['TP_ESCOLA'].value_counts(normalize=True)*100).round(1))

In [ ]:

mapa_esc = {2:'Pública', 3:'Privada'}
df_esc = df[df['TP_ESCOLA'].isin([2,3])].copy()
df_esc['Escola'] = df_esc['TP_ESCOLA'].map(mapa_esc)

print(f"Total com escola declarada: {len(df_esc):,} ({len(df_esc)/len(df)*100:.1f}% do total)")
print(f"\nQuebra por tipo:")
print(df_esc['Escola'].value_counts())

## 1. Nota média por tipo de escola — comparação 2019 vs 2023

In [ ]:

medias = df_esc.groupby(['NU_ANO','Escola'])['NOTA_MEDIA'].agg(['mean','median','std','count']).round(1)
print("Estatísticas da nota média por ano e tipo de escola:\n")
print(medias)

In [ ]:

fig, ax = plt.subplots(figsize=(9,5))
sns.barplot(data=df_esc, x='NU_ANO', y='NOTA_MEDIA', hue='Escola',
            palette=['#4C72B0','#DD8452'], ax=ax, errorbar=None)
ax.set_title('Nota média por tipo de escola — ENEM 2019 vs 2023')
ax.set_xlabel('Ano')
ax.set_ylabel('Nota média')
plt.tight_layout()
plt.show()

In [ ]:

fig, ax = plt.subplots(figsize=(10,5))
sns.boxplot(data=df_esc, x='NU_ANO', y='NOTA_MEDIA', hue='Escola',
            palette=['#4C72B0','#DD8452'], ax=ax, showfliers=False)
ax.set_title('Distribuição da nota média por tipo de escola e ano')
ax.set_xlabel('Ano')
ax.set_ylabel('Nota média')
plt.tight_layout()
plt.show()

## 2. Comparação RS vs SP

In [ ]:

tabela = df_esc.groupby(['NU_ANO','SG_UF_PROVA','Escola'])['NOTA_MEDIA'].mean().round(1).unstack()
print("Nota média por ano, estado e tipo de escola:\n")
print(tabela)

In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)

for ax, ano in zip(axes, [2019, 2023]):
    sub = df_esc[df_esc['NU_ANO']==ano]
    sns.barplot(data=sub, x='SG_UF_PROVA', y='NOTA_MEDIA', hue='Escola',
                palette=['#4C72B0','#DD8452'], ax=ax, errorbar=None)
    ax.set_title(f'ENEM {ano}')
    ax.set_xlabel('Estado')
    ax.set_ylabel('Nota média')

plt.tight_layout()
plt.show()

## 3. Gap pública × privada por área de conhecimento

A diferença entre escolas públicas e privadas é maior em qual área?

In [ ]:
areas_label = {'NU_NOTA_CN':'Ciências da Natureza', 'NU_NOTA_CH':'Ciências Humanas',
               'NU_NOTA_LC':'Linguagens', 'NU_NOTA_MT':'Matemática',
               'NU_NOTA_REDACAO':'Redação'}

dados_area = df_esc.melt(id_vars=['Escola'], value_vars=list(areas_label.keys()),
                         var_name='Area', value_name='Nota')
dados_area['Area'] = dados_area['Area'].map(areas_label)
dados_area = dados_area.dropna(subset=['Nota'])


gap = dados_area.groupby(['Area','Escola'])['Nota'].mean().unstack()
gap['Diferença Privada–Pública'] = (gap['Privada'] - gap['Pública']).round(1)
print(gap.sort_values('Diferença Privada–Pública', ascending=False))

In [ ]:

fig, ax = plt.subplots(figsize=(11,5))
sns.barplot(data=dados_area, y='Area', x='Nota', hue='Escola',
            palette=['#4C72B0','#DD8452'], ax=ax, errorbar=None,
            order=['Redação','Matemática','Ciências da Natureza','Ciências Humanas','Linguagens'])
ax.set_title('Nota média por área de conhecimento e tipo de escola')
ax.set_xlabel('Nota média')
ax.set_ylabel('')
plt.tight_layout()
plt.show()

## 4. Síntese dos achados

Registre aqui os principais achados observados. Sugestões do que descrever:

- Qual a diferença geral entre pública e privada? Aumentou ou diminuiu de 2019 para 2023?
- Há diferença entre RS e SP em como o tipo de escola afeta o desempenho?
- Em qual área o gap pública × privada é maior?
- Você nota algum padrão inesperado nos dados?